# LLM Manager SDK Quickstart

`llm-manager` 를 **순수 SDK manager** 로 사용하는 가이드 (별도 서버 없음).

**구조:**
```
이 노트북 (agent) — llm-manager 는 같은 프로세스 SDK
┌───────────────────────────────────────────────────┐
│ register_or_get(bytes) ── file-manager(8600) →      │ 등록 → file_path
│                                                     │
│ llm_request_async(stored_files=[StoredFile])        │  같은 프로세스에서 handler 직접 조합:
│   → MDB 캐시 조회 (mdb-manager)                      │
│   → miss 면 file-manager 에서 bytes lazy fetch       │
│   → LLM 호출 (llm-handler → Gemini)                 │
│   → 결과 MDB 저장 + (result, meta) 반환              │
└───────────────────────────────────────────────────┘
```

에이전트는 **file-manager SDK 의 `register_or_get`** 으로 등록 후 받은 `file_path + 5 메타`
(= `StoredFile`) 만 `llm_request_async(stored_files=[...])` 에 넘긴다. bytes 는 file-manager 에만
적재되고 llm-manager 는 leader 1 명만 fetch (standby dedup).

**주요 변경점 (라이브러리 → SDK):**

| | 라이브러리 (v1) | SDK (현재) |
|--|---------------|------------|
| 설정 | URL 3개 + configure 3번 | 불필요 (handler URL 하드코딩, 첫 호출 시 자동 초기화) |
| 파일 전달 | `file_paths=["report.pdf"]` | `stored_files=[StoredFile(...)]` (file-manager `register_or_get` 결과) |
| 시간 설정 | `ttl_seconds=`, `context_cache_ttl=` 각각 | `timings={"mdb_ttl": ..., "context_cache_ttl": ...}` |


In [1]:
# 설치 — llm-manager + 의존 manager 들 editable 설치
# (노트북 CWD = llm-manager/quickstarts/llm_manager 기준 상대경로)
# llm-handler SDK 는 llm-manager 에 흡수됨 — 별도 설치 불필요.
%pip install -q \
    -e ../../../mdb-manager \
    -e ../../../redis-manager \
    -e ../../../file-manager \
    -e ../.. \
    nest_asyncio


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 설정

In [1]:
import os
import nest_asyncio

from pathlib import Path
from dotenv import load_dotenv

nest_asyncio.apply()

# .env 는 커널이 자동으로 안 읽음 → Kiwoom-Manager/.env 를 명시 로드 (handler URL 전부).
# override=True : .env 값이 기존 env(원격 기본값 등)를 덮어쓰도록.
load_dotenv(Path.cwd().parents[2] / ".env", override=True)

os.environ["MONGODB_DATABASE"] = "kw_ai_dev"   # 테스트 전용 DB
TEST_DB = os.environ["MONGODB_DATABASE"]

# 확인 — 전부 localhost 로 떠야 함
for k in ("MDB_HANDLER_URL", "LLM_HANDLER_URL", "FILE_HANDLER_URL", "REDIS_URL", "MONGODB_DATABASE"):
    print(f"  {k} = {os.environ.get(k)}")

  MDB_HANDLER_URL = http://localhost:8300
  LLM_HANDLER_URL = http://localhost:8100
  FILE_HANDLER_URL = http://localhost:8600
  REDIS_URL = redis://192.7.9.45:7617/0
  MONGODB_DATABASE = kw_ai_dev


In [2]:
import os
import nest_asyncio
nest_asyncio.apply()

# file-manager 는 노트북이 직접 호출하므로 URL 설정 필요 (llm_manager 보다 먼저 import 됨).
# llm/mdb/redis handler URL 은 llm_manager 가 첫 호출 시 자동 설정.

TEST_DB = "kw_ai_dev"

print(f"TEST_DB = {TEST_DB}")

TEST_DB = kw_ai_dev


## 타이밍 유틸리티

각 단계별 소요 시간과 네트워크 흐름을 추적하는 헬퍼.

In [3]:
import time
from contextlib import contextmanager
from pathlib import Path

from file_manager import FileManagerHandler
from llm_manager import StoredFile

fm = FileManagerHandler()  # Singleton; build_metadata는 로컬 계산 (HTTP 없음)

@contextmanager
def timer(label: str):
    """구간별 소요 시간 측정."""
    start = time.perf_counter()
    print(f"  [{label}] 시작...")
    yield
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  [{label}] 완료: {elapsed:.0f}ms")

def print_flow(title: str, meta, elapsed_ms: float):
    """요청 흐름 요약 출력."""
    m = meta.model_dump() if hasattr(meta, "model_dump") else meta
    source = m.get("source", "?")
    cache_hit = m.get("cache_hit", False)
    cache_key = m.get("cache_key", "")[:16]

    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  소스:      {source} ({'MDB 캐시 히트' if cache_hit else 'LLM 호출'})")
    print(f"  캐시 키:   {cache_key}...")
    print(f"  총 소요:   {elapsed_ms:.0f}ms")

    if m.get("route_reason"):
        print(f"  경로:      {m['route_reason']}")
    print()

async def build_stored_file(path) -> StoredFile:
    """로컬 파일 → file-manager 등록 → StoredFile (6 필드).

    register_or_get 이 hash 3 필드 (file_name/file_size/file_mtime) 로 check_cache →
    miss 면 upload_file. bytes 는 file-manager 에만 적재되고 caller (이 노트북) 는 이후 release 가능.
    같은 파일 재호출 시 register_or_get 이 hit 으로 빠져 업로드 생략.
    """
    p = Path(path)
    sf_dict = await fm.register_or_get(
        bytes_data=p.read_bytes(),
        metadata=fm.build_metadata(
            file_name=p.name,
            file_mtime=int(p.stat().st_mtime),
            file_size=p.stat().st_size,
        ),
    )
    return StoredFile(**sf_dict)


---
## 1. 기본 호출 — 텍스트만 (파일 없음)

```
llm_request_async({llm_input}, cache_mode=True)
  → MDB 캐시 조회 (miss)
  → LLM 호출 (llm-handler → Gemini)
  → MDB 캐시 저장 → (result, meta)
```

In [4]:
from llm_manager import llm_request_async

# --- 1회차: LLM 직접 호출 (캐시 miss) ---
start = time.perf_counter()
result, meta = await llm_request_async(
    {
        "contents": "1+1은?",
        "model": "gemini-2.5-flash",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("1회차: 첫 호출 (캐시 miss → LLM)", meta, elapsed)
print(f"  응답: {(result.text or '')[:100]}")


  1회차: 첫 호출 (캐시 miss → LLM)
  소스:      cache (MDB 캐시 히트)
  캐시 키:   81c7c9133a10d5e9...
  총 소요:   139ms

  응답: 1+1은 **2**입니다.


In [5]:
# --- 2회차: 동일 입력 → MDB 캐시 히트 (LLM 호출 없음) ---
start = time.perf_counter()
result2, meta2 = await llm_request_async(
    {
        "contents": "1+1은?",
        "model": "gemini-2.5-flash",
    },
    database="kw_ai_dev",
)
elapsed2 = (time.perf_counter() - start) * 1000

print_flow("2회차: 동일 입력 (MDB 캐시 히트)", meta2, elapsed2)
print(f"  응답: {(result2.text or '')[:100]}")
print(f"  속도 비교: {elapsed:.0f}ms → {elapsed2:.0f}ms ({elapsed/max(elapsed2,0.1):.1f}x 빠름)")


  2회차: 동일 입력 (MDB 캐시 히트)
  소스:      cache (MDB 캐시 히트)
  캐시 키:   81c7c9133a10d5e9...
  총 소요:   13ms

  응답: 1+1은 **2**입니다.
  속도 비교: 139ms → 13ms (10.5x 빠름)


---
## 2. 파일 포함 호출

```
1. build_stored_file (file-manager 등록)
     register_or_get(bytes, meta) → 6 필드 StoredFile
2. llm_request_async(stored_files=[..])
     → resolve_stored_files (placeholder)
     → MDB 캐시 조회 → miss
     → leader 분기 lazy fetch (file-manager)
     → LLM 호출 (Gemini) → MDB 저장 → (result, meta)
```


In [6]:
from llm_manager import llm_request_async

test_file = "small_file.pdf"
if not Path(test_file).exists():
    print(f"⚠️ {test_file}이 없습니다. 아래 셀은 건너뛰세요.")
else:
    print(f"파일: {test_file} ({Path(test_file).stat().st_size / 1024:.0f}KB)")
    print()

    # 1. 로컬 파일 → file-manager 등록 → StoredFile 구성
    with timer("build_stored_file"):
        sf = await build_stored_file(test_file)
    print(f"    → mime={sf.mime_type}, size={sf.file_size}, file_path={sf.file_path}")
    print()

    # 2. llm_request_async (stored_files= 전달)
    start = time.perf_counter()
    with timer("llm_request_async 전체"):
        result, meta = await llm_request_async(
            {
                "contents": "이 문서를 한 문장으로 요약해줘",
                "model": "gemini-2.5-flash",
            },
            stored_files=[sf],
            database=TEST_DB,
        )
    elapsed = (time.perf_counter() - start) * 1000

    print()
    print_flow("파일 포함 호출 (1회차)", meta, elapsed)
    print(f"  응답: {(result.text or '')[:200]}")


파일: small_file.pdf (598KB)

  [build_stored_file] 시작...
  [build_stored_file] 완료: 96ms
    → mime=application/pdf, size=612269, file_path=ea3ceca9681d4cf3b238b792a33bc886.pdf

  [llm_request_async 전체] 시작...
  [llm_request_async 전체] 완료: 11565ms


  파일 포함 호출 (1회차)
  소스:      live (LLM 호출)
  캐시 키:   a1c177d88af4bfdc...
  총 소요:   11565ms

  응답: 이 2024 KB금융그룹 지속가능경영보고서는 투자자 관점에서 기업의 지속가능성 관련 거버넌스, 전략, 위험관리, 지표 및 목표를 IFRS 공시기준에 따라 제시하며, 리스크 관리, 기후위기 대응, 포용금융, 금융소비자보호, 디지털 혁신 및 기술 등 핵심 중대 이슈에 대한 정보를 담고 있습니다.


---
## 3. done_check + fetch_by_cache_key

캐시 존재 여부만 확인하고, 필요 시 직접 조회.

```
done_check({llm_input})        → MDB 캐시 조회 → {cache_key, cache_hit, cached}
fetch_by_cache_key(cache_key)  → MDB 에서 결과 반환 → (result, meta)
```

In [7]:
from llm_manager import done_check, fetch_by_cache_key

# --- done_check: 캐시 존재 여부 확인 ---
with timer("done_check"):
    dc = await done_check(
        {"contents": "1+1은?", "model": "gemini-2.5-flash"},
        database="kw_ai_dev",
    )

print(f"  cache_key: {dc['cache_key'][:16]}...")
print(f"  cache_hit: {dc['cache_hit']}")
print()

# --- fetch_by_cache_key: 캐시 키로 직접 조회 ---
if dc["cache_hit"]:
    with timer("fetch_by_cache_key"):
        result, meta = await fetch_by_cache_key(
            dc["cache_key"],
            database="kw_ai_dev",
        )
    print(f"  응답: {(result.text or '')[:100]}")
else:
    print("  캐시 없음 — 위의 1번 셀을 먼저 실행하세요")

  [done_check] 시작...
  [done_check] 완료: 14ms
  cache_key: 81c7c9133a10d5e9...
  cache_hit: True

  [fetch_by_cache_key] 시작...
  [fetch_by_cache_key] 완료: 10ms
  응답: 1+1은 **2**입니다.


---
## 4. 기존 라이브러리와 비교

| | 라이브러리 (v1) | SDK (현재) |
|--|---------------|------------|
| **설정** | MDB_HANDLER_URL + LLM_HANDLER_URL + REDIS_URL + configure 3개 | 불필요 (handler URL 하드코딩, 첫 호출 시 자동 초기화) |
| **의존성** | llm-handler, mdb-handler, redis-handler 전부 | `llm_manager` 패키지 (handler 들은 그 의존성) |
| **사용 코드** | `llm_request_async(...)` | `llm_request_async(...)` (동일) |
| **파일 전달** | `file_paths=["report.pdf"]` | `stored_files=[StoredFile(...)]` (file-manager `register_or_get` 결과) |
| **파일 등록** | SDK → Redis/Gemini 직접 처리 | agent → file-manager `register_or_get` (llm-manager 는 lazy fetch) |
| **캐시 조회** | SDK가 MDB 직접 조회 | llm-manager(SDK)가 mdb-manager 통해 조회 |
| **시간 설정** | `ttl_seconds=`, `context_cache_ttl=` 각각 | `timings={"mdb_ttl": ..., "context_cache_ttl": ...}` |


---
## 5. 시스템 프롬프트 + task_name

In [ ]:
start = time.perf_counter()
result, meta = await llm_request_async(
    {
        "contents": "ESG 평가 기준을 알려줘",
        "model": "gemini-2.5-flash",
        "system": "너는 ESG 전문가야. 한국어로 답변해.",
        "temperature": 0,
        "max_tokens": 200,
        "task_name": "esg_agent",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("시스템 프롬프트 + task_name", meta, elapsed)
print(f"  응답: {(result.text or '')[:200]}")

---
## 6. 대용량 파일 (>50MB PDF → 자동 split + Files API)

```
1. build_stored_file (file-manager 등록)
2. llm_request_async(stored_files=[..])
     → MDB 캐시 조회 → miss
     → classify_files → Files API 경로
     → leader 분기 lazy fetch (file-manager)
     → split_pdf (50MB 초과 분할) → Files API 업로드
     → LLM 호출 → MDB 저장 → (result, meta)
```


In [9]:
from pathlib import Path

LARGE_PDF = "large_file.pdf"

if not Path(LARGE_PDF).exists():
    print(f"⚠️ 대용량 PDF를 찾을 수 없습니다: {LARGE_PDF}")
    print("   경로를 수정하거나 이 셀을 건너뛰세요.")
else:
    size_mb = Path(LARGE_PDF).stat().st_size / (1024 * 1024)
    print(f"파일: {Path(LARGE_PDF).name} ({size_mb:.1f}MB)")
    print()

    # 1. 로컬 파일 → file-manager 등록 → StoredFile 구성
    with timer("build_stored_file"):
        sf_large = await build_stored_file(LARGE_PDF)
    print(f"    → mime={sf_large.mime_type}, file_path={sf_large.file_path}")
    print()

    # 2. 첫 호출 (서비스 내부에서 lazy fetch + split + upload + LLM)
    start = time.perf_counter()
    with timer("대용량 PDF 전체"):
        result, meta = await llm_request_async(
            {
                "contents": "이 PDF의 제목을 한글 한 줄로.",
                "model": "gemini-2.5-flash",
                "max_tokens": 1000,
            },
            stored_files=[sf_large],
            database=TEST_DB,
        )
    elapsed = (time.perf_counter() - start) * 1000

    print_flow("대용량 PDF (1회차)", meta, elapsed)
    print(f"  응답: {(result.text or '')[:200]}")


파일: large_file.pdf (88.4MB)

  [build_stored_file] 시작...
  [build_stored_file] 완료: 368ms
    → mime=application/pdf, file_path=ce38dc1f792141dbbe1e33203ca7bcf6.pdf

  [대용량 PDF 전체] 시작...
  [대용량 PDF 전체] 완료: 11537ms

  대용량 PDF (1회차)
  소스:      live (LLM 호출)
  캐시 키:   1ecc8de631efc965...
  총 소요:   11537ms

  응답: 2025 GS건설 지속가능경영보고서


In [10]:
# 6-2. 같은 파일 재호출 → MDB 캐시 히트 (lazy fetch / 업로드 / LLM 호출 모두 없음)
if Path(LARGE_PDF).exists():
    start = time.perf_counter()
    result2, meta2 = await llm_request_async(
        {
            "contents": "이 PDF의 제목을 한글 한 줄로.",
            "model": "gemini-2.5-flash",
            "max_tokens": 1000,
        },
        stored_files=[sf_large],
        database=TEST_DB,
    )
    elapsed2 = (time.perf_counter() - start) * 1000

    print_flow("대용량 PDF (2회차 — MDB 캐시 히트)", meta2, elapsed2)
    print(f"  응답: {(result2.text or '')[:200]}")
    print(f"  속도 비교: {elapsed:.0f}ms → {elapsed2:.0f}ms")



  대용량 PDF (2회차 — MDB 캐시 히트)
  소스:      cache (MDB 캐시 히트)
  캐시 키:   1ecc8de631efc965...
  총 소요:   15ms

  응답: 2025 GS건설 지속가능경영보고서
  속도 비교: 11537ms → 15ms


---
## 7. Context Cache 만료/연장 — `delete_context_cache_async` / `update_context_cache_async`

`context_cache_ttl` 을 길게 잡아두고, task 가 끝나면 `delete_context_cache_async` 로 **무조건 5초 뒤** 정리해 storage 비용을 줄인다 (sync 래퍼 `delete_context_cache` 도 있음). 더 오래 살려야 하면 `update_context_cache_async(cache_id, ttl=...)` 로 연장한다.

두 함수 모두 `context_cache_id`(hash_key)를 받는다 — `meta.context_cache_id`. (Gemini `cachedContents/...` 이름은 받지 않음 → `ValueError`.)

```
agent ─→ llm_request_async(timings={"context_cache_ttl": 3600})
              ↓ 응답
         meta.context_cache_id  ── 식별자 저장
              ↓ task 종료
agent ─→ delete_context_cache_async([cache_id])   # 5초 뒤 만료
         (또는 update_context_cache_async(cache_id, ttl=3600) 로 연장)
```

In [ ]:
import uuid
from llm_manager import delete_context_cache_async, update_context_cache_async

if not Path(LARGE_PDF).exists() or "sf_large" not in dir():
    print("⚠️ §6 셀이 먼저 실행되어야 합니다.")
else:
    # MDB cache miss 보장 — live LLM 호출 + 새 Context Cache 생성
    unique = uuid.uuid4().hex[:8]

    start = time.perf_counter()
    with timer("context_cache 생성"):
        result_cc, meta_cc = await llm_request_async(
            {
                "contents": f"이 문서의 한 줄 요약 ({unique})",
                "model": "gemini-2.5-flash",
            },
            stored_files=[sf_large],
            timings={"context_cache_ttl": 3600},  # 1h 넉넉히
            database=TEST_DB,
        )
    elapsed = (time.perf_counter() - start) * 1000

    print()
    print(f"  cache_id: {meta_cc.context_cache_id}")
    print(f"  응답: {(result_cc.text or '')[:120]}")
    print(f"  elapsed: {elapsed:.0f}ms")

    if meta_cc.context_cache_id:
        # (선택) 수명 연장 — ttl 사용자가 지정 (1~3600초)
        with timer("update_context_cache_async"):
            ext = await update_context_cache_async(meta_cc.context_cache_id, ttl=3600)
        print(f"  연장 expired={ext.expired} failed={ext.failed}")

        # task 끝났다고 가정 — 무조건 5초 뒤 만료 (시간 인자 없음)
        with timer("delete_context_cache_async"):
            release = await delete_context_cache_async([meta_cc.context_cache_id])
        print(f"  expired: {release.expired}")
        print(f"  failed:  {release.failed}")
    else:
        print()
        print("  ⚠️ context_cache_id 없음 — MDB cache HIT 됐을 수 있음")

---
## 7. 반환값 구조

모든 `llm_request_async` 함수는 `(result, meta)` 튜플을 반환합니다.

`result`는 `LLMResponse`, `meta`는 `ResponseMeta` — 모두 Pydantic 모델이므로 속성으로 접근:

```python
# result: LLMResponse
result.text           # str | None
result.model          # str
result.usage          # TokenUsage | None
result.usage.input_tokens   # int
result.usage.output_tokens  # int
result.usage.total_tokens   # int
result.finish_reason  # str | None
result.request_id     # str
result.function_calls # list[FunctionCall] | None

# meta: ResponseMeta
meta.source      # str — "live" (LLM 직접 호출) | "cache" (MDB 캐시)
meta.cache_hit   # bool
meta.cache_key   # str | None — SHA-256 hex (64자)
meta.metadata    # dict — {"service_name": "...", "task_name": "...", "model": "..."}
                 #   service_name = env SERVICE_NAME(실행 주체), task_name = llm_input의 호출 라벨
meta.collection  # str | None
meta.route_reason  # str | None — debug=True 시 포함
```

### 캐시 키 동작 원리

`llm_input`에서 14개 필드 전부 추출(미입력/None → `""`) → 값 내부 정규화 → 키 정렬·JSON 직렬화 → SHA-256 해시

- **같은 입력 = 같은 캐시 키** → 캐시 히트
- **파일 첨부 시**: 파일 내용의 checksum이 캐시 키에 포함 → 같은 파일 + 같은 질문 = 캐시 히트 (파일명/URI 무관)

---
# Appendix: 에이전트 개발 패턴

`llm_request_async`를 사용하는 에이전트 개발 시 자주 쓰는 패턴입니다.

## 패턴 A: 단건 처리

단건 LLM 호출 + 캐시. 에이전트 내 단일 LLM 호출의 기본 패턴.

In [ ]:
from llm_manager import llm_request_async

start = time.perf_counter()
result_a, meta_a = await llm_request_async(
    {
        "contents": "삼성전자 반도체 사업 현황을 두 줄로 요약해줘.",
        "model": "gemini-2.5-flash",
        "temperature": 0,
        "max_tokens": 150,
        "task_name": "fund_comment",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("패턴 A: 단건 처리", meta_a, elapsed)
print(f"  응답: {(result_a.text or '')[:200]}")
if result_a.usage:
    u = result_a.usage
    print(f"  토큰: {u.input_tokens} in / {u.output_tokens} out")

---
## 패턴 B: N건 병렬 — `llm_request_async` + 타이밍 로그

펀드마다 **DB 조회 지연**을 다르게 두어 LLM 요청 시각이 어긋나게 합니다.
로그에 `[경과초] 코드 DB완료` → `LLM시작` → `LLM끝(소요초)` 가 찍혀, **비동기 병렬 진행**을 확인할 수 있습니다.

```
005930: task → DB(즉시) → [LLM 2~4s] ─────────────┐
000660: task → DB(0.4s) → [LLM ...]              ├─ as_completed
035420: task → DB(0.8s) → [LLM ...]              │
068270: task → DB(1.2s) → [LLM ...]              ┘
```

In [ ]:
import asyncio
import time

from llm_manager import llm_request_async

FUNDS = [
    {"code": "005930", "name": "삼성전자",  "sector": "반도체"},
    {"code": "000660", "name": "SK하이닉스", "sector": "반도체"},
    {"code": "035420", "name": "NAVER",      "sector": "IT"},
    {"code": "068270", "name": "셀트리온",   "sector": "바이오"},
]

DB_STAGGER_SEC = 0.42


async def process_fund(fund: dict, idx: int, wall0: float) -> dict:
    """DB 조회(시뮬) 후 LLM 호출."""
    await asyncio.sleep(DB_STAGGER_SEC * idx)
    t_db_done = time.perf_counter() - wall0
    print(f"  [{t_db_done:.2f}s] {fund['code']} DB 완료")

    t_llm_start = time.perf_counter() - wall0
    result, meta = await llm_request_async(
        {
            "contents": (
                f"{fund['name']}({fund['code']})의 {fund['sector']} 사업 현황을 "
                f"투자 관점에서 간결한 한 문단(3~4문장)으로 설명해줘."
            ),
            "model": "gemini-2.5-flash",
            "temperature": 0.3,
            "max_tokens": 300,
            "task_name": "fund_comment",
        },
        database="kw_ai_dev",
    )
    t_llm_end = time.perf_counter() - wall0
    llm_sec = t_llm_end - t_llm_start

    print(f"  [{t_llm_end:.2f}s] {fund['code']} LLM 끝 ({llm_sec:.2f}s, {meta.source})")

    return {
        "code": fund["code"],
        "name": fund["name"],
        "text": (result.text or ""),
        "source": meta.source,
        "t_db_done": t_db_done,
        "t_llm_start": t_llm_start,
        "t_llm_end": t_llm_end,
        "llm_sec": llm_sec,
    }


# 실행
wall0 = time.perf_counter()
tasks = [asyncio.create_task(process_fund(f, i, wall0)) for i, f in enumerate(FUNDS)]
many_results = []
for coro in asyncio.as_completed(tasks):
    many_results.append(await coro)

total = time.perf_counter() - wall0
print(f"\n  전체 소요: {total:.2f}s (순차 시 ~{sum(r['llm_sec'] for r in many_results):.2f}s)")

### 결과 확인 — 요약 표

`LLM시작` 시각이 DB끝 직후에 연달아 찍히면 비동기 병렬 진행 확인.

In [14]:
rows = sorted(many_results, key=lambda x: x["code"])
hdr = f"{'코드':<8} {'DB끝(s)':>8} {'LLM시작(s)':>10} {'LLM끝(s)':>9} {'순수LLM(s)':>10} {'source':<8}"
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['code']:<8} "
        f"{r['t_db_done']:8.2f} "
        f"{r['t_llm_start']:10.2f} "
        f"{r['t_llm_end']:9.2f} "
        f"{r['llm_sec']:10.2f} "
        f"{r['source']:<8}"
    )

print()
for r in rows:
    preview = r["text"].replace("\n", " ")[:100]
    print(f"[{r['code']}] {preview}")

코드         DB끝(s)   LLM시작(s)   LLM끝(s)   순수LLM(s) source  
----------------------------------------------------------
000660       0.42       0.42      0.48       0.06 cache   
005930       0.00       0.00      0.01       0.01 cache   
035420       0.84       0.84      0.85       0.01 cache   
068270       1.26       1.26      1.28       0.01 cache   

[000660] SK하이닉스는 인공지능(AI
[005930] 삼성전자(005930)의
[035420] 네이버(035420)의
[068270] 셀트리온은 바이오시밀러 분야


---
## 정리

테스트 데이터를 삭제합니다. (kw_ai_dev DB)

In [ ]:
# MDB 캐시 정리 — done_check으로 cache_key를 구해서 확인
from llm_manager import done_check

test_inputs = [
    {"contents": "1+1은?", "model": "gemini-2.5-flash"},
    {"contents": "ESG 평가 기준을 알려줘", "model": "gemini-2.5-flash", "system": "너는 ESG 전문가야. 한국어로 답변해.", "temperature": 0, "max_tokens": 200, "task_name": "esg_agent"},
    {"contents": "삼성전자 반도체 사업 현황을 두 줄로 요약해줘.", "model": "gemini-2.5-flash", "temperature": 0, "max_tokens": 150, "task_name": "fund_comment"},
]

# 펀드 패턴 B 입력도 추가
for fund in FUNDS:
    test_inputs.append({
        "contents": (
            f"{fund['name']}({fund['code']})의 {fund['sector']} 사업 현황을 "
            f"투자 관점에서 간결한 한 문단(3~4문장)으로 설명해줘."
        ),
        "model": "gemini-2.5-flash",
        "temperature": 0.3,
        "max_tokens": 300,
        "task_name": "fund_comment",
    })

print("캐시 키 조회 중...")
for inp in test_inputs:
    dc = await done_check(inp, database="kw_ai_dev")
    status = "hit" if dc["cache_hit"] else "miss"
    print(f"  {dc['cache_key'][:16]}... ({status})")

print("\n정리 완료 (실제 삭제는 MDB에서 직접 수행)")